# NB 4.1 &mdash; Regressió logística: el primer classificador

**MP 5134** &mdash; UT4

*Dades: pingüins de l'arxipèlag Palmer.*

---
### Què farem avui

Fins ara els nostres models donaven un número: una massa, un preu, una demanda
elèctrica. Avui en farem un que **pren una decisió**: és d'aquesta classe o no?
Això és el que hi ha darrere de molts sistemes que fas servir cada dia, com el
filtre de correu brossa o l'alerta de pagament fraudulent del banc.

Avui parlarem poc i treballareu molt. Hi ha tres pauses «Ara tu» llargues per fer
en grup, i cadascuna acaba comparant els vostres resultats amb els d'un altre grup.

Quan acabis hauries de saber respondre aquestes preguntes:

- Quins passos cal seguir per entrenar un classificador i fer-lo servir?
- Què vol dir que el model digui «75%»?
- Què és la frontera de decisió i com es mou amb el llindar?
- Per què un 86% d'encerts pot no ser gran cosa?

## 1. El problema: és un Adelie?

Imagina un equip de recerca a l'Antàrtida. Pot mesurar el bec d'un pingüí amb un
regle, però no sempre sap de quina espècie és. Volem un model que, a partir de les
mesures, respongui: **és un Adelie, sí o no?**

Quan la resposta és una categoria i no una quantitat, parlem de **classificació**,
i les possibles respostes són les **classes**. El model que farem servir es diu
**regressió logística**: malgrat el nom, classifica.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from sklearn.dummy import DummyClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Còpia de les dades de palmerpenguins (Gorman et al., 2014) al repositori del mòdul
URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT01-Entorn_de_treball_primer_model/penguins/penguins.csv"

VARIABLES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
COLORS = ["#0072B2", "#D55E00", "#009E73"]

df = pd.read_csv(URL_DADES).dropna(subset=VARIABLES)
df["es_adelie"] = (df["species"] == "Adelie").astype(int)

sns.scatterplot(data=df, x="bill_length_mm", y="bill_depth_mm",
                hue="es_adelie", style="es_adelie", palette=COLORS[:2])
plt.title("Els Adelie tenen el bec curt i alt")
plt.show()

Són les dades de la UT1, sense els dos pingüins que no tenen mesures. La columna
`es_adelie` és la resposta que volem predir: 1 vol dir que sí i 0, que no. Els colors
de `COLORS` es distingeixen també amb daltonisme, i per això els farem servir a tots
els gràfics.

Els Adelie queden a dalt a l'esquerra, i amb un regle quasi podríem traçar una línia
que els separés de la resta. **Classificar és això: trobar aquesta línia i fer-la servir
amb pingüins nous.**

## 2. La recepta per classificar

Sempre seguirem els mateixos cinc passos. Valdran per a tots els classificadors del
curs, també els de la UT5 i la UT6.

| Pas | Què fem |
|---|---|
| 1. Preparar | Triem les columnes `X` i la resposta `y` |
| 2. Partir | Separem entrenament i prova, amb les classes ben repartides |
| 3. Entrenar | `fit` amb les dades d'entrenament |
| 4. Comparar | Encerts del model contra els d'un model de referència |
| 5. Predir | Fem servir el model amb casos nous |

In [ ]:
COLUMNES = ["bill_length_mm", "bill_depth_mm"]
X = df[COLUMNES]
y = df["es_adelie"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

print(f"Adelie a l'entrenament: {y_train.mean():.1%} de {len(y_train)} pingüins")
print(f"Adelie a la prova:      {y_test.mean():.1%} de {len(y_test)} pingüins")

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

referencia = DummyClassifier(strategy="most_frequent")
referencia.fit(X_train, y_train)

print(f"Encerts del model:      {model.score(X_test, y_test):.1%}")
print(f"Encerts de referència:  {referencia.score(X_test, y_test):.1%}")

La primera novetat és `stratify=y`, que fa una **partició estratificada**: conserva la
mateixa proporció de cada classe a les dues parts, aquí un 44%. Com més rara és una
classe, més important és, perquè l'atzar podria deixar-ne molt pocs casos a la prova.
A partir d'ara, quan classifiquem, sempre estratificarem.

La resta ve de la UT1: `fit`, `score` i el `Dummy`. En classificació, `score` és el
percentatge d'encerts (en anglès, *accuracy*). `max_iter` són les voltes que pot fer
el model per ajustar-se, i 5.000 n'hi ha de sobres.

El model encerta 83 de 86 pingüins, un 96,5%. La referència, que diu sempre «no és
Adelie», en encerta un 55,8%. La diferència és clara: **el model ha après**.

Ara ve el més útil, el pas 5: fer-lo servir amb pingüins nous.

In [ ]:
nous = pd.DataFrame({"bill_length_mm": [38, 50, 42.5],
                     "bill_depth_mm": [19, 15, 18]})

print("Decisió:", model.predict(nous))
print("Probabilitat de ser Adelie:", model.predict_proba(nous)[:, 1].round(3))

`predict` dona la decisió i `predict_proba`, la **probabilitat** de pertànyer a cada
classe. El `[:, 1]` n'agafa la columna de la classe 1, la que ens interessa.

Els dos primers pingüins són clars. El tercer té un 75% de ser Adelie: el model diu
que sí, però amb menys seguretat. Quan un classificador s'ha de fer servir de debò,
aquesta probabilitat sovint és tan útil com la decisió.

---
### Ara tu · Exercici 1: a la recerca dels Chinstrap

*Uns 35 minuts, en grup.*

Ara l'equip vol localitzar els Chinstrap, que només són un de cada cinc pingüins.

**a)** Crea la columna `es_chinstrap` i segueix la recepta amb les dues columnes del
bec. Quants encerts fa el model? I la referència? Compareu-ho amb el cas dels Adelie:
què ha canviat?

**b)** Hi ha quatre variables, que fan sis parelles possibles. Amb la funció `provar` de
la cel·la següent, que segueix la recepta i n'escriu el resultat, repartiu-vos les
parelles i ompliu la taula:

```python
provar(["bill_depth_mm", "body_mass_g"], "es_chinstrap")
```

| Parella de variables | Encerts del model | Encerts de referència |
|---|---|---|
| | | |

Quina parella és la millor? N'hi ha cap de pitjor que la referència? Què vol dir això?

**c)** Dibuixeu els punts de la millor parella i de la pitjor, canviant `x`, `y` i `hue`
al gràfic de la secció 1. Es veu per què una funciona i l'altra no?

**d)** Entreneu un model amb la millor parella i inventeu mesures fins a trobar un
pingüí que el faci dubtar, amb una probabilitat entre el 40% i el 60%.

**Compareu amb un altre grup.** Heu triat la mateixa parella? Si un model fa gairebé un
99% amb 86 pingüins, quants errors són? Què li diríeu a l'equip de recerca: us en fiaríeu?

In [ ]:
def provar(columnes, objectiu, C=1, llavor=42):
    # Segueix la recepta i escriu els encerts del model i de la referència
    X_train, X_test, y_train, y_test = train_test_split(
        df[columnes], df[objectiu], test_size=0.25,
        stratify=df[objectiu], random_state=llavor
    )
    # Més voltes que a la recepta: amb quatre variables sense escalar, de vegades en calen
    model = LogisticRegression(C=C, max_iter=20000).fit(X_train, y_train)
    referencia = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
    print(f"{columnes}, C={C}, llavor={llavor}")
    print(f"  Model: {model.score(X_test, y_test):.1%}   Referència: {referencia.score(X_test, y_test):.1%}")

In [ ]:
# Escriu aquí el teu codi

## 3. Com decideix el model

El model no respon directament sí o no: primer calcula una probabilitat i després
la talla. Vegem-ho amb una sola variable, la llargada del bec.

In [ ]:
llargada = X_train[["bill_length_mm"]]
model_1 = LogisticRegression(max_iter=5000).fit(llargada, y_train)

plt.scatter(llargada, model_1.predict_proba(llargada)[:, 1], color=COLORS[0])
plt.axhline(0.5, color="black", linestyle="--")
plt.xlabel("bill_length_mm")
plt.ylabel("Probabilitat de ser Adelie")
plt.show()

Aquesta corba en forma de S és la **funció logística**, i és l'únic que cal retenir-ne:
converteix qualsevol mesura en una probabilitat entre 0 i 1. Amb un bec curt, el model
està segur que és Adelie; amb un de llarg, que no. La línia discontínua és el
**llindar**: per sobre del 50%, el model diu que sí.

Amb dues variables passa el mateix, i el lloc on la probabilitat val un 50% forma una
línia sobre el pla. És la **frontera de decisió**.

In [ ]:
DecisionBoundaryDisplay.from_estimator(
    model, X_train, response_method="predict",
    cmap=ListedColormap(COLORS[:2]), alpha=0.2
)
sns.scatterplot(data=X_test, x="bill_length_mm", y="bill_depth_mm",
                hue=y_test, style=y_test, palette=COLORS[:2])
plt.title("Frontera de decisió i pingüins de prova")
plt.show()

Cada zona de color és el que el model respon en aquell lloc, i els punts són els
pingüins de prova. Els errors són els punts que queden a la zona de l'altre color.
La frontera d'una regressió logística sempre està feta de línies rectes; a la UT5 i la
UT6 en veurem de més flexibles.

El llindar del 50% no és obligatori: el podem triar nosaltres.

In [ ]:
probabilitat = model.predict_proba(X_test)[:, 1]

for llindar in [0.2, 0.5, 0.95]:
    marcats = probabilitat >= llindar
    encertats = (marcats & (y_test == 1)).sum()
    falses_alarmes = (marcats & (y_test == 0)).sum()
    perduts = (~marcats & (y_test == 1)).sum()
    print(f"Llindar {llindar}: marcats {marcats.sum()}, encertats {encertats}, "
          f"falses alarmes {falses_alarmes}, perduts {perduts}")

Amb un llindar baix (20%), el model marca més pingüins: només se li escapa un Adelie,
però dona una falsa alarma. Amb un llindar alt (95%), no dona cap falsa alarma, però
se'n perden sis. **El model calcula la probabilitat, però on tallar és una decisió
nostra**, i depèn de quin error ens surt més car. Hi posarem nom al NB 4.3.

---
### Ara tu · Exercici 2: una alarma per als Chinstrap

*Uns 35 minuts, en grup.*

Cada pingüí que el model marca com a Chinstrap s'ha de tornar a capturar per fer-li
una analítica. Feu servir el model de l'exercici 1 amb **les dues columnes del bec**,
que s'equivoca prou perquè es notin les decisions.

**a)** Dibuixeu la seva frontera de decisió amb els pingüins de prova, i després la del
model amb la millor parella que heu trobat. Què canvia?

**b)** Per al model del bec, ompliu aquesta taula. La cel·la del llindar de la secció 3
només necessita uns petits canvis.

| Llindar | Marcats | Encertats | Falses alarmes | Chinstrap perduts |
|---|---|---|---|---|
| 0,2 | | | | |
| 0,3 | | | | |
| 0,5 | | | | |
| 0,7 | | | | |
| 0,8 | | | | |

**c)** Trieu un llindar per a cada situació i justifiqueu-lo en una o dues frases:

- **Situació A.** La captura estressa molt l'animal i l'equip no vol molestar cap
  pingüí sense motiu.
- **Situació B.** S'està fent un cens dels Chinstrap i l'equip vol trobar-ne tants com
  pugui, encara que hagi de capturar alguns pingüins de més.

**Compareu amb un altre grup.** Heu triat el mateix llindar? Hi ha un llindar correcte?

*Si acabeu abans:* a la frontera, canvieu `response_method` per `"predict_proba"` i
`cmap` per `"RdBu"`. Què mostra ara el fons?

In [ ]:
# Escriu aquí el teu codi

## 4. Llegir el model: coeficients i C

Com a la regressió lineal, podem mirar què ha après el model.

In [ ]:
pd.Series(model.coef_[0], index=COLUMNES).round(2)

Del coeficient, primer mirem el **signe**. La llargada és negativa: com més llarg és
el bec, menys probable és que sigui Adelie. L'alçada és positiva: com més alt, més
probable. És el mateix que vèiem al primer gràfic. Aquí podem comparar-ne la mida
perquè totes dues es mesuren en mil·límetres; amb unitats diferents caldria escalar
primer, com al NB 3.2.

La regressió logística té un paràmetre important, **C**, que controla quanta llibertat
té el model per ajustar-se a les dades d'entrenament. Amb una C molt petita, el model és
molt prudent; amb una C gran, s'ajusta molt, i aquest és el camí cap al sobreajust del
NB 2.3. El valor per defecte, `C=1`, sol anar bé per començar. A l'exercici següent en
veureu l'efecte, i a la UT7 aprendrem a triar-lo amb mètode.

---
### Ara tu · Exercici 3: el classificador complet

*Uns 40 minuts, en grup.*

L'equip ja no vol respostes de sí o no: vol un model que digui **l'espècie** de cada
pingüí. La recepta és exactament la mateixa, i només canvia `y`, que ara és la columna
`species`. El model tria entre les tres classes sense que li hàgim de dir res més.

**a)** Entreneu el model amb les dues columnes del bec. Quants encerts fa, i quants la
referència? Inventeu un pingüí i mireu què retorna `predict_proba`. Amb
`model.classes_` sabreu de quina espècie és cada columna.

**b)** Dibuixeu les tres zones. Aquest cop, el gràfic necessita que li indiquem l'ordre de les classes:

```python
DecisionBoundaryDisplay.from_estimator(
    model, X_train, response_method="predict",
    cmap=ListedColormap(COLORS), alpha=0.2
)
sns.scatterplot(data=X_test, x="bill_length_mm", y="bill_depth_mm", hue=y_test,
                style=y_test, hue_order=model.classes_, palette=COLORS)
```

**c)** Amb `provar`, ompliu la taula. Podeu escriure, per exemple,
`provar(COLUMNES, "species", C=0.01)`.

| Columnes | C=0,0001 | C=0,01 | C=1 | C=100 |
|---|---|---|---|---|
| Les 2 del bec | | | | |
| Bec i aleta (3) | | | | |
| Les 4 variables | | | | |

**d)** Trieu el model que donaríeu a l'equip i torneu a provar-lo amb
`llavor=7` i `llavor=0`. El resultat es manté?

**e)** Escriviu-ne la **fitxa**, en quatre o cinc frases: quines columnes fa servir, quina
C, quants encerts fa i contra quina referència, i en què creieu que es pot equivocar.

**Compareu amb un altre grup.** Intercanvieu les fitxes. Quin model triaríeu, el vostre
o el seu? Té sentit fer servir quatre variables si amb menys ja n'hi ha prou?

In [ ]:
# Escriu aquí el teu codi

## 5. El que ens emportem

Ja saps fer un classificador de principi a fi: preparar, partir amb `stratify`, entrenar,
comparar amb la referència i predir. El model calcula una **probabilitat**, el **llindar**
la converteix en una decisió, i la **frontera** mostra on canvia la resposta. I un número
d'encerts no diu res sol: amb els Chinstrap, la referència ja feia un 80%.

Al NB 4.2 farem servir un altre classificador amb la mateixa recepta, el k-NN, i al
NB 4.3 aprendrem a mesurar millor els errors que avui hem comptat a mà.

## 6. Per al debat de classe

Penseu un classificador que seria útil al vostre entorn: a la feina, a l'institut o en
una afició. Quines serien les classes? Quines dades tindríeu? Què faria la persona que
rep la predicció? I quin error us preocuparia més: una falsa alarma o un cas que se us
escapa?